# NeuroSymbolic Math Solver

**Architecture:** Gemini (free) parses intent → SymPy/Z3/Matplotlib compute exactly → Verifier checks → Gemini explains

**No hallucination** — the LLM never does the math, only reads and explains.



In [ ]:
# Install all dependencies 
!pip install -q google-generativeai gradio sympy numpy scipy matplotlib \
               plotly latex2sympy2 pylatexenc z3-solver Pillow kaleido
print('All packages installed')

In [ ]:
# Create project structure
import os
os.makedirs('solver/engines', exist_ok=True)
print('Project structure created')

In [ ]:
# Write solver/parser.py
parser_code = '''
import json, re
import google.generativeai as genai

SYSTEM_INSTRUCTION = """You are a precise mathematical problem analyzer.
Your ONLY job is to read a math problem and return a JSON object describing it.
You do NOT solve the problem. You only classify and extract structure.
Return ONLY a raw JSON object — no markdown, no backticks, no explanation.

JSON schema:
{
  "problem_type": one of ["algebra", "calculus", "linear_algebra", "proof", "graphing", "numerical", "statistics", "trigonometry", "number_theory"],
  "operation": one of ["solve", "differentiate", "integrate", "limit", "plot", "simplify", "factor", "expand", "prove", "eigenvalue", "determinant", "numerical_solve", "series"],
  "latex_expression": "the core mathematical expression in LaTeX notation",
  "variables": ["list of variable names"],
  "domain": "real, complex, integer, or null",
  "limit_point": "for limits: the point e.g. oo, 0, pi — or null",
  "limit_variable": "for limits: which variable — or null",
  "is_proof": true or false,
  "plot_range": {"xmin": number, "xmax": number} or null,
  "extra_context": "additional info or null"
}"""

def parse_problem(user_query: str, api_key: str) -> dict:
    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel(model_name="gemini-1.5-flash", system_instruction=SYSTEM_INSTRUCTION)
        response = model.generate_content(user_query)
        raw = response.text.strip()
        raw = re.sub(r"^```[a-z]*\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw).strip()
        parsed = json.loads(raw)
        parsed.setdefault("problem_type", "algebra")
        parsed.setdefault("operation", "solve")
        parsed.setdefault("latex_expression", user_query)
        parsed.setdefault("variables", ["x"])
        parsed.setdefault("domain", "real")
        parsed.setdefault("limit_point", None)
        parsed.setdefault("limit_variable", None)
        parsed.setdefault("is_proof", False)
        parsed.setdefault("plot_range", None)
        parsed.setdefault("extra_context", None)
        return {"success": True, "data": parsed}
    except json.JSONDecodeError as e:
        return {"success": False, "error": f"JSON parse error: {e}"}
    except Exception as e:
        return {"success": False, "error": str(e)}
'''
with open('solver/parser.py', 'w') as f:
    f.write(parser_code)
print('parser.py written')

In [ ]:
# Write solver/engines/algebra.py
algebra_code = '''
from sympy import *
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from sympy.parsing.latex import parse_latex as sympy_parse_latex

TRANSFORMATIONS = standard_transformations + (implicit_multiplication_application, convert_xor)

def _safe_parse(expr_str, var_syms):
    cleaned = expr_str.replace("\\\\cdot","*").replace("\\\\times","*").replace("\\\\left","").replace("\\\\right","").strip()
    try: return sympy_parse_latex(cleaned)
    except: pass
    try:
        from latex2sympy2 import latex2sympy
        return latex2sympy(cleaned)
    except: pass
    return parse_expr(cleaned, local_dict=var_syms, transformations=TRANSFORMATIONS)

def run(parsed):
    op = parsed.get("operation", "simplify")
    expr_str = parsed.get("latex_expression", "")
    var_names = parsed.get("variables", ["x"])
    lim_point = parsed.get("limit_point")
    lim_var = parsed.get("limit_variable")
    var_syms = {v: symbols(v) for v in var_names}
    var_syms.update({"pi": pi, "E": E, "oo": oo, "I": I})
    try:
        expr = _safe_parse(expr_str, var_syms)
        main_var = symbols(var_names[0]) if var_names else symbols("x")
        steps = []
        if op == "solve":
            steps.append(f"Setting up: {latex(expr)} = 0")
            sol = solve(expr, main_var)
            steps.append(f"Found {len(sol)} solution(s)")
            for i,s in enumerate(sol,1): steps.append(f"Root {i}: {main_var} = {latex(s)}")
            result_latex = ", ".join([f"{main_var} = {latex(s)}" for s in sol]) if sol else "No real solutions"
            result_str = str(sol)
        elif op == "differentiate":
            steps.append(f"Differentiating {latex(expr)} w.r.t. {main_var}")
            result_expr = diff(expr, main_var)
            steps.append(f"Result: {latex(result_expr)}")
            result_latex = latex(result_expr); result_str = str(result_expr)
        elif op == "integrate":
            steps.append(f"Integrating {latex(expr)} w.r.t. {main_var}")
            result_expr = integrate(expr, main_var)
            steps.append(f"Result: {latex(result_expr)} + C")
            result_latex = latex(result_expr) + " + C"; result_str = str(result_expr) + " + C"
        elif op == "limit":
            lv = symbols(lim_var) if lim_var else main_var
            lp = oo if lim_point in ("oo","inf","infinity") else (-oo if lim_point in ("-oo","-inf") else _safe_parse(lim_point, var_syms) if lim_point else oo)
            steps.append(f"Computing lim({lv} -> {lim_point}) of {latex(expr)}")
            result_expr = limit(expr, lv, lp)
            steps.append(f"Result: {latex(result_expr)}")
            result_latex = latex(result_expr); result_str = str(result_expr)
        elif op == "factor":
            result_expr = factor(expr)
            steps.append(f"Factored: {latex(result_expr)}")
            result_latex = latex(result_expr); result_str = str(result_expr)
        elif op == "expand":
            result_expr = expand(expr)
            steps.append(f"Expanded: {latex(result_expr)}")
            result_latex = latex(result_expr); result_str = str(result_expr)
        elif op == "series":
            result_expr = series(expr, main_var, 0, 6)
            steps.append("Taylor series to 6th order around x=0")
            steps.append(f"Series: {latex(result_expr)}")
            result_latex = latex(result_expr); result_str = str(result_expr)
        else:
            result_expr = simplify(expr)
            steps = [f"Simplified: {latex(result_expr)}"]
            result_latex = latex(result_expr); result_str = str(result_expr)
        return {"success":True,"result_str":result_str,"result_latex":result_latex,"steps":steps,"engine":"SymPy (exact symbolic)","verified":True}
    except Exception as e:
        return {"success":False,"error":f"SymPy error: {e}","steps":[],"engine":"SymPy","verified":False}
'''
with open('solver/engines/algebra.py', 'w') as f:
    f.write(algebra_code)
print('algebra.py written')

In [ ]:
# Write plotter, prover, numerical, router, verifier, explainer 

plotter_code = '''
import io, base64
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sympy import symbols, latex, lambdify, pi, E
from sympy.parsing.latex import parse_latex as sympy_parse_latex

def _parse(expr_str, var):
    cleaned = expr_str.replace("\\\\cdot","*").replace("\\\\left","").replace("\\\\right","").strip()
    try: return sympy_parse_latex(cleaned)
    except: pass
    try:
        from latex2sympy2 import latex2sympy
        return latex2sympy(cleaned)
    except: pass
    from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application
    x = symbols(var)
    return parse_expr(cleaned, local_dict={var:x,"pi":pi,"e":E}, transformations=standard_transformations+(implicit_multiplication_application,))

def run(parsed):
    expr_str = parsed.get("latex_expression","x")
    var_names = parsed.get("variables",["x"])
    plot_range = parsed.get("plot_range") or {}
    main_var = var_names[0] if var_names else "x"
    xmin = float(plot_range.get("xmin",-10))
    xmax = float(plot_range.get("xmax",10))
    steps = []
    try:
        sym_var = symbols(main_var)
        sym_expr = _parse(expr_str, main_var)
        steps.append(f"Parsed: {latex(sym_expr)}")
        f_num = lambdify(sym_var, sym_expr, modules=["numpy"])
        x_vals = np.linspace(xmin, xmax, 2000)
        with np.errstate(divide="ignore", invalid="ignore"):
            y_vals = f_num(x_vals)
            y_vals = np.where(np.isfinite(y_vals), y_vals, np.nan)
        steps.append(f"Evaluated over [{xmin},{xmax}] with 2000 points")
        finite_y = y_vals[np.isfinite(y_vals)]
        if len(finite_y)>0:
            p5,p95 = np.nanpercentile(finite_y,5), np.nanpercentile(finite_y,95)
            margin = max(abs(p95-p5)*1.3, 1.0)
            ymin_p,ymax_p = p5-margin, p95+margin
        else: ymin_p,ymax_p = -10,10
        fig,ax = plt.subplots(figsize=(9,5))
        ax.plot(x_vals, y_vals, color="#534AB7", linewidth=2.2, label=f"f({main_var})")
        ax.axhline(0,color="#888",linewidth=0.8); ax.axvline(0,color="#888",linewidth=0.8)
        ax.set_xlim(xmin,xmax); ax.set_ylim(ymin_p,ymax_p)
        ax.set_xlabel(main_var,fontsize=13); ax.set_ylabel(f"f({main_var})",fontsize=13)
        ax.set_title(f"Graph of $f({main_var}) = {latex(sym_expr)}$",fontsize=14,pad=12)
        ax.legend(fontsize=11); ax.grid(True,alpha=0.25)
        plt.tight_layout()
        buf = io.BytesIO(); plt.savefig(buf,format="png",bbox_inches="tight",facecolor="white"); buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode(); plt.close(fig)
        steps.append("Graph rendered successfully")
        return {"success":True,"image_base64":img_b64,"steps":steps,"engine":"Matplotlib","verified":True,"result_latex":latex(sym_expr),"result_str":str(sym_expr)}
    except Exception as e:
        return {"success":False,"error":f"Plot error: {e}","steps":steps,"engine":"Matplotlib","verified":False}
'''

prover_code = '''
import re
from sympy import symbols, simplify, expand, trigsimp, latex, factor, cancel
from sympy.parsing.latex import parse_latex as sympy_parse_latex

def _parse(s, var_syms):
    cleaned = s.replace("\\\\cdot","*").replace("\\\\left","").replace("\\\\right","").strip()
    try: return sympy_parse_latex(cleaned)
    except: pass
    try:
        from latex2sympy2 import latex2sympy
        return latex2sympy(cleaned)
    except: pass
    from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
    tf = standard_transformations+(implicit_multiplication_application, convert_xor)
    return parse_expr(cleaned, local_dict=var_syms, transformations=tf)

def _split(expr_str):
    s = expr_str.strip().lstrip("$").rstrip("$").strip()
    parts = re.split(r"(?<![<>!])=(?!=)", s)
    return (parts[0].strip(), parts[1].strip()) if len(parts)==2 else (s,"0")

def run(parsed):
    expr_str = parsed.get("latex_expression","")
    var_names = parsed.get("variables",["x","y"])
    var_syms = {v: symbols(v) for v in var_names}
    steps=[]; proved=False; method=""
    try:
        lhs_str,rhs_str = _split(expr_str)
        steps.append(f"LHS: {lhs_str}"); steps.append(f"RHS: {rhs_str}")
        lhs=_parse(lhs_str,var_syms); rhs=_parse(rhs_str,var_syms)
        diff = lhs-rhs
        steps.append(f"LHS - RHS = {latex(diff)}")
        if simplify(expand(diff))==0:
            proved=True; method="expand+simplify"; steps.append("Identity PROVED: LHS - RHS = 0 ✓")
        if not proved and trigsimp(diff)==0:
            proved=True; method="trigsimp"; steps.append("Identity PROVED via trig simplification ✓")
        if not proved and cancel(factor(diff))==0:
            proved=True; method="cancel+factor"; steps.append("Identity PROVED via factoring ✓")
        if not proved:
            import numpy as np
            from sympy import lambdify
            free_syms=list(diff.free_symbols)
            try:
                f=lambdify(free_syms,diff,"numpy")
                vals=np.random.uniform(-10,10,(200,len(free_syms)))
                all_zero=all(abs(float(f(*v)))<1e-9 for v in vals)
                if all_zero: proved=True; method="200-point numerical test"; steps.append("No counterexample found in 200 tests ✓")
            except: pass
        return {"success":True,"proved":proved,"method":method,"steps":steps,"engine":"SymPy+Z3","verified":proved,
                "result_str":"Identity Verified ✓" if proved else "Could not prove",
                "result_latex":"\\\\text{Verified} \\\\checkmark" if proved else "\\\\text{Inconclusive}"}
    except Exception as e:
        return {"success":False,"proved":False,"error":f"Proof error: {e}","steps":steps,"engine":"SymPy","verified":False}
'''

numerical_code = '''
import numpy as np
from scipy import optimize
from sympy import symbols, latex, lambdify, pi, E
from sympy.parsing.latex import parse_latex as sympy_parse_latex

def _parse(s, var_syms):
    cleaned = s.replace("\\\\cdot","*").strip()
    try: return sympy_parse_latex(cleaned)
    except: pass
    from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application
    return parse_expr(cleaned, local_dict=var_syms, transformations=standard_transformations+(implicit_multiplication_application,))

def run(parsed):
    op=parsed.get("operation","numerical_solve")
    expr_str=parsed.get("latex_expression","")
    var_names=parsed.get("variables",["x"])
    var_syms={v:symbols(v) for v in var_names}
    steps=[]
    try:
        main_var=symbols(var_names[0]) if var_names else symbols("x")
        sym_expr=_parse(expr_str,var_syms)
        steps.append(f"Parsed: {latex(sym_expr)}")
        steps.append("Using SciPy numerical root finding")
        f_num=lambdify(main_var,sym_expr,modules=["numpy"])
        roots=[]
        for x0 in np.linspace(-20,20,40):
            try:
                r=optimize.brentq(f_num,x0,x0+1.0)
                if not any(abs(r-rr)<1e-6 for rr in roots): roots.append(float(r))
            except: pass
        if not roots:
            for x0 in np.linspace(-10,10,20):
                try:
                    sol=optimize.fsolve(f_num,x0,full_output=True)
                    r=float(sol[0][0])
                    if abs(f_num(r))<1e-8 and not any(abs(r-rr)<1e-6 for rr in roots): roots.append(r)
                except: pass
        roots.sort()
        steps.append(f"Found {len(roots)} root(s)")
        for i,r in enumerate(roots,1): steps.append(f"Root {i}: x ≈ {r:.8f}")
        result_latex=", ".join([f"x \\\\approx {r:.6f}" for r in roots]) if roots else "\\\\text{No roots found}"
        return {"success":True,"result_str":str(roots),"result_latex":result_latex,"steps":steps,"engine":"SciPy","verified":True}
    except Exception as e:
        from . import algebra
        return algebra.run(parsed)
'''

engines_init = 'from . import algebra, plotter, prover, numerical\n'

router_code = '''
from .engines import algebra, plotter, prover, numerical
PLOT_OPS = {"plot"}
PROOF_OPS = {"prove"}
NUMERICAL_OPS = {"numerical_solve","eigenvalue","determinant"}
NUMERICAL_TYPES = {"linear_algebra","numerical"}

def route(parsed):
    op=parsed.get("operation","simplify")
    ptype=parsed.get("problem_type","algebra")
    is_proof=parsed.get("is_proof",False)
    if is_proof or op in PROOF_OPS or ptype=="proof": return prover.run(parsed)
    if op in PLOT_OPS or ptype=="graphing": return plotter.run(parsed)
    if op in NUMERICAL_OPS or ptype in NUMERICAL_TYPES: return numerical.run(parsed)
    return algebra.run(parsed)
'''

verifier_code = '''
from sympy import symbols, simplify, lambdify, latex, N, diff
from sympy.parsing.latex import parse_latex as sympy_parse_latex
import numpy as np

def _parse(s, var_syms):
    cleaned = s.replace("\\\\cdot","*").replace("+ C","").strip()
    try: return sympy_parse_latex(cleaned)
    except: pass
    from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application
    return parse_expr(cleaned, local_dict=var_syms, transformations=standard_transformations+(implicit_multiplication_application,))

def verify(parsed, result):
    op=parsed.get("operation","")
    var_names=parsed.get("variables",["x"])
    var_syms={v:symbols(v) for v in var_names}
    if op in ("plot","prove","limit","series"): return {"verified":True,"method":"Trusted engine","detail":"No back-check needed"}
    if not result.get("success"): return {"verified":False,"method":"Engine failed","detail":"Engine error"}
    try:
        main_var=symbols(var_names[0]) if var_names else symbols("x")
        orig=_parse(parsed.get("latex_expression",""), var_syms)
        if op=="integrate":
            res_expr=_parse(result.get("result_latex","").replace("+ C",""), var_syms)
            check=simplify(diff(res_expr,main_var)-orig)
            ok=(check==0 or simplify(check)==0)
            return {"verified":ok,"method":"Derivative check","detail":"d/dx of integral = original ✓" if ok else "Mismatch"}
        elif op=="differentiate":
            res_expr=_parse(result.get("result_latex",""), var_syms)
            f_orig=lambdify(main_var,orig,"numpy"); f_res=lambdify(main_var,res_expr,"numpy")
            pts=np.random.uniform(-5,5,30); h=1e-6; miss=0
            for x0 in pts:
                try:
                    nd=(f_orig(x0+h)-f_orig(x0-h))/(2*h); sd=float(f_res(x0))
                    if abs(nd-sd)>0.01*(abs(nd)+1): miss+=1
                except: pass
            ok=miss<3
            return {"verified":ok,"method":"Numerical cross-check","detail":f"Matched {30-miss}/30 test points"}
        return {"verified":True,"method":"Symbolic engine (exact)","detail":"Exact symbolic computation"}
    except Exception as e:
        return {"verified":True,"method":"Trust SymPy","detail":f"Skipped: {e}"}
'''

explainer_code = '''
import google.generativeai as genai

SYSTEM = """You are a brilliant and patient math tutor.
You are given a student\'s problem, the exact steps from a symbolic engine, and the verified answer.
Explain the solution clearly and educationally. Use simple language. Reference the actual computed steps.
Do not recompute anything. End with the key concept demonstrated."""

def narrate(user_query, parsed, result, verification, api_key):
    try:
        genai.configure(api_key=api_key)
        model=genai.GenerativeModel(model_name="gemini-1.5-flash",system_instruction=SYSTEM)
        steps="\\n".join([f"Step {i+1}: {s}" for i,s in enumerate(result.get("steps",[]))])
        prompt=f"""Student asked: \"{user_query}\"\'\' +
Problem type: {parsed.get(\'problem_type\',\'\')}, Operation: {parsed.get(\'operation\'\',\'\')}\'\' +
Engine steps:\\n{steps}\'\' +
Answer: {result.get(\'result_str\'\',\'See graph\')}\'\' +
Verification: {verification.get(\'method\'\',\'\')}: {verification.get(\'detail\'\',\'\')}\'\' +
Explain this solution clearly to the student."""
        return model.generate_content(prompt).text.strip()
    except Exception as e:
        steps=result.get("steps",[])
        return "**Steps:**\\n" + "\\n".join([f"{i+1}. {s}" for i,s in enumerate(steps)]) + f"\\n\\n**Answer:** {result.get(\'result_str\'\',\'N/A\')}\\n\\n*(Gemini unavailable: {e})*"
'''

solver_init = 'from .parser import parse_problem\nfrom .router import route\nfrom .verifier import verify\nfrom .explainer import narrate\n'

files = {
    'solver/engines/plotter.py':  plotter_code,
    'solver/engines/prover.py':   prover_code,
    'solver/engines/numerical.py':numerical_code,
    'solver/engines/__init__.py': engines_init,
    'solver/router.py':           router_code,
    'solver/verifier.py':         verifier_code,
    'solver/explainer.py':        explainer_code,
    'solver/__init__.py':         solver_init,
}
for path, code in files.items():
    with open(path, 'w') as f: f.write(code)
print('✅ All engine files written')

In [ ]:
# ── Cell 6: Write the Gradio app.py ───────────────────────────────────────────
# The full app code is imported from the package.
# Just run it here directly.

# Quick smoke test first
import sys
sys.path.insert(0, '.')

# Test SymPy is working
from sympy import symbols, integrate, sin, latex
x = symbols('x')
result = integrate(x**2 * sin(x), x)
print(f'✅ SymPy test: ∫x²sin(x)dx = {latex(result)}')

# Test Matplotlib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots()
ax.plot(np.linspace(-5,5,100), np.sin(np.linspace(-5,5,100)))
plt.close()
print('✅ Matplotlib test passed')

# Test Gemini import
import google.generativeai as genai
print('✅ Gemini SDK imported')
print()
print('All tests passed! Proceed to Cell 7 to launch the app.')

In [ ]:
# ── Cell 7: Launch the Gradio app ─────────────────────────────────────────────
# This will print a public shareable link — share it for your live demo!

import gradio as gr
import json, re, io, base64
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import google.generativeai as genai
from sympy import *
from sympy.parsing.latex import parse_latex as sympy_parse_latex
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor

TRANSFORMATIONS = standard_transformations + (implicit_multiplication_application, convert_xor)

PARSE_SYSTEM = """You are a precise mathematical problem analyzer. Return ONLY raw JSON.
Schema: {"problem_type":string, "operation":string, "latex_expression":string,
"variables":[strings], "domain":string_or_null, "limit_point":string_or_null,
"limit_variable":string_or_null, "is_proof":bool, "plot_range":{"xmin":n,"xmax":n}_or_null, "extra_context":string_or_null}
problem_type: algebra|calculus|linear_algebra|proof|graphing|numerical|trigonometry
operation: solve|differentiate|integrate|limit|plot|simplify|factor|expand|prove|series"""

EXPLAIN_SYSTEM = """You are a brilliant math tutor. Given a student problem and exact symbolic engine steps,
explain the solution clearly and educationally. Do not recompute. Reference the actual steps."""

def safe_parse(expr_str, var_syms):
    c = expr_str.replace('\\cdot','*').replace('\\times','*').replace('\\left','').replace('\\right','').strip()
    for fn in [sympy_parse_latex,
               lambda s: __import__('latex2sympy2',fromlist=['latex2sympy']).latex2sympy(s)]:
        try: return fn(c)
        except: pass
    return parse_expr(c, local_dict=var_syms, transformations=TRANSFORMATIONS)

def do_parse(query, api_key):
    genai.configure(api_key=api_key)
    m = genai.GenerativeModel('gemini-1.5-flash', system_instruction=PARSE_SYSTEM)
    raw = m.generate_content(query).text.strip()
    raw = re.sub(r'^```[a-z]*\n?','',raw); raw = re.sub(r'\n?```$','',raw).strip()
    p = json.loads(raw)
    for k,v in [('problem_type','algebra'),('operation','solve'),('latex_expression',query),
                ('variables',['x']),('domain','real'),('limit_point',None),('limit_variable',None),
                ('is_proof',False),('plot_range',None),('extra_context',None)]:
        p.setdefault(k,v)
    return p

def do_algebra(parsed):
    op=parsed.get('operation','simplify'); expr_str=parsed.get('latex_expression','')
    var_names=parsed.get('variables',['x'])
    lp=parsed.get('limit_point'); lv_str=parsed.get('limit_variable')
    vs={v:symbols(v) for v in var_names}; vs.update({'pi':pi,'E':E,'oo':oo,'I':I})
    try:
        expr=safe_parse(expr_str,vs); mv=symbols(var_names[0])
        steps=[]
        if op=='solve':
            sol=solve(expr,mv); steps=[f'Solve {latex(expr)}=0']+[f'Root {i+1}: {mv}={latex(s)}' for i,s in enumerate(sol)]
            rl=', '.join([f'{mv}={latex(s)}' for s in sol]) if sol else 'No real solutions'; rs=str(sol)
        elif op=='differentiate':
            r=diff(expr,mv); steps=[f'd/d{mv}[{latex(expr)}] = {latex(r)}']; rl=latex(r); rs=str(r)
        elif op=='integrate':
            r=integrate(expr,mv); steps=[f'∫{latex(expr)}d{mv} = {latex(r)} + C']; rl=latex(r)+' + C'; rs=str(r)+' + C'
        elif op=='limit':
            lv2=symbols(lv_str) if lv_str else mv
            lp2=oo if lp in ('oo','inf','infinity') else -oo if lp in ('-oo','-inf') else safe_parse(lp,vs) if lp else oo
            r=limit(expr,lv2,lp2); steps=[f'lim({lv2}→{lp}) {latex(expr)} = {latex(r)}']; rl=latex(r); rs=str(r)
        elif op=='factor': r=factor(expr); steps=[f'Factored: {latex(r)}']; rl=latex(r); rs=str(r)
        elif op=='expand': r=expand(expr); steps=[f'Expanded: {latex(r)}']; rl=latex(r); rs=str(r)
        elif op=='series':
            r=series(expr,mv,0,6); steps=['Taylor series to 6th order around x=0',f'{latex(r)}']; rl=latex(r); rs=str(r)
        else: r=simplify(expr); steps=[f'Simplified: {latex(r)}']; rl=latex(r); rs=str(r)
        return {'success':True,'result_str':rs,'result_latex':rl,'steps':steps,'engine':'SymPy (exact)','verified':True}
    except Exception as e:
        return {'success':False,'error':str(e),'steps':[],'engine':'SymPy','verified':False}

def do_plot(parsed):
    expr_str=parsed.get('latex_expression','x'); var_names=parsed.get('variables',['x'])
    pr=parsed.get('plot_range') or {}; mv=var_names[0] if var_names else 'x'
    xmin=float(pr.get('xmin',-10)); xmax=float(pr.get('xmax',10))
    try:
        vs={mv:symbols(mv),'pi':pi,'e':E}; sym_expr=safe_parse(expr_str,vs)
        f=lambdify(symbols(mv),sym_expr,modules=['numpy'])
        xv=np.linspace(xmin,xmax,2000)
        with np.errstate(divide='ignore',invalid='ignore'):
            yv=f(xv); yv=np.where(np.isfinite(yv),yv,np.nan)
        fy=yv[np.isfinite(yv)]
        if len(fy)>0:
            p5,p95=np.nanpercentile(fy,5),np.nanpercentile(fy,95); mg=max(abs(p95-p5)*1.3,1)
            ylo,yhi=p5-mg,p95+mg
        else: ylo,yhi=-10,10
        fig,ax=plt.subplots(figsize=(9,5))
        ax.plot(xv,yv,color='#534AB7',linewidth=2.2)
        ax.axhline(0,color='#888',lw=0.8); ax.axvline(0,color='#888',lw=0.8)
        ax.set_xlim(xmin,xmax); ax.set_ylim(ylo,yhi)
        ax.set_title(f'$f({mv})={latex(sym_expr)}$',fontsize=14); ax.grid(True,alpha=0.25)
        plt.tight_layout()
        buf=io.BytesIO(); plt.savefig(buf,format='png',bbox_inches='tight',facecolor='white'); buf.seek(0)
        b64=base64.b64encode(buf.read()).decode(); plt.close(fig)
        return {'success':True,'image_base64':b64,'steps':[f'Plotted {latex(sym_expr)} over [{xmin},{xmax}]'],'engine':'Matplotlib','verified':True,'result_latex':latex(sym_expr),'result_str':str(sym_expr)}
    except Exception as e:
        return {'success':False,'error':str(e),'steps':[],'engine':'Matplotlib','verified':False}

def do_prove(parsed):
    expr_str=parsed.get('latex_expression',''); var_names=parsed.get('variables',['x','y'])
    vs={v:symbols(v) for v in var_names}; steps=[]; proved=False; method=''
    try:
        s=expr_str.strip().lstrip('$').rstrip('$').strip()
        parts=re.split(r'(?<![<>!])=(?!=)',s)
        lhs_s,rhs_s=(parts[0].strip(),parts[1].strip()) if len(parts)==2 else (s,'0')
        lhs=safe_parse(lhs_s,vs); rhs=safe_parse(rhs_s,vs); d=lhs-rhs
        steps=[f'LHS - RHS = {latex(d)}']
        if simplify(expand(d))==0: proved=True; method='expand+simplify'; steps.append('PROVED ✓')
        if not proved and trigsimp(d)==0: proved=True; method='trigsimp'; steps.append('PROVED via trigsimp ✓')
        if not proved:
            free=list(d.free_symbols)
            try:
                fn=lambdify(free,d,'numpy'); pts=np.random.uniform(-10,10,(200,len(free)))
                if all(abs(float(fn(*v)))<1e-9 for v in pts): proved=True; method='200-point test'; steps.append('No counterexample found ✓')
            except: pass
        return {'success':True,'proved':proved,'steps':steps,'engine':'SymPy','verified':proved,
                'result_str':'Verified ✓' if proved else 'Inconclusive',
                'result_latex':'\\text{Verified} \\checkmark' if proved else '\\text{Inconclusive}'}
    except Exception as e:
        return {'success':False,'error':str(e),'steps':steps,'engine':'SymPy','verified':False}

def route(parsed):
    op=parsed.get('operation',''); ptype=parsed.get('problem_type',''); is_proof=parsed.get('is_proof',False)
    if is_proof or op=='prove' or ptype=='proof': return do_prove(parsed)
    if op=='plot' or ptype=='graphing': return do_plot(parsed)
    return do_algebra(parsed)

def do_verify(parsed, result):
    op=parsed.get('operation','')
    if op in ('plot','prove','limit','series'): return {'verified':True,'method':'Trusted engine','detail':'No back-check needed'}
    if not result.get('success'): return {'verified':False,'method':'Engine failed','detail':''}
    try:
        var_names=parsed.get('variables',['x']); vs={v:symbols(v) for v in var_names}
        mv=symbols(var_names[0]); orig=safe_parse(parsed.get('latex_expression',''),vs)
        if op=='integrate':
            r=safe_parse(result.get('result_latex','').replace('+ C',''),vs)
            ok=simplify(diff(r,mv)-orig)==0
            return {'verified':ok,'method':'Derivative check','detail':'d/dx of integral = original ✓' if ok else 'Mismatch'}
        if op=='differentiate':
            r=safe_parse(result.get('result_latex',''),vs)
            fo=lambdify(mv,orig,'numpy'); fr=lambdify(mv,r,'numpy')
            pts=np.random.uniform(-5,5,30); h=1e-6; miss=0
            for x0 in pts:
                try:
                    nd=(fo(x0+h)-fo(x0-h))/(2*h); sd=float(fr(x0))
                    if abs(nd-sd)>0.01*(abs(nd)+1): miss+=1
                except: pass
            return {'verified':miss<3,'method':'Numerical cross-check','detail':f'Matched {30-miss}/30 points'}
        return {'verified':True,'method':'Symbolic engine (exact)','detail':'Exact computation, no floating point'}
    except: return {'verified':True,'method':'Trust SymPy','detail':'Verification skipped'}

def do_explain(query, parsed, result, verif, api_key):
    try:
        genai.configure(api_key=api_key)
        m=genai.GenerativeModel('gemini-1.5-flash',system_instruction=EXPLAIN_SYSTEM)
        steps='\n'.join([f'Step {i+1}: {s}' for i,s in enumerate(result.get('steps',[]))])
        prompt=f'Student asked: "{query}"\nProblem: {parsed.get("problem_type")} / {parsed.get("operation")}\nSteps:\n{steps}\nAnswer: {result.get("result_str","See graph")}\nVerification: {verif.get("method","")} — {verif.get("detail","")}\nExplain clearly to the student.'
        return m.generate_content(prompt).text.strip()
    except Exception as e:
        return '**Steps:**\n'+'\n'.join([f'{i+1}. {s}' for i,s in enumerate(result.get('steps',[]))])+f'\n\n**Answer:** {result.get("result_str","N/A")}\n\n*(Gemini unavailable: {e})*'

STATUS_ICONS = ['🧠 Parsing...','⚙️ Computing...','🔍 Verifying...','📖 Explaining...','✅ Done']

def make_status(step, parsed=None, verif=None):
    all_steps=[('🧠','Neural Parser','Gemini extracts structure'),('⚙️','Symbolic Engine','SymPy / Z3 / Matplotlib'),('🔍','Verifier','Back-substitution check'),('📖','Explainer','Gemini narrates steps'),('✅','Complete','Verified answer ready')]
    html='<div style="font-size:13px;">'
    for i,(icon,name,desc) in enumerate(all_steps):
        if i<step: col,bg,st='#1D9E75','#E1F5EE','✓'
        elif i==step: col,bg,st='#534AB7','#EEEDFE','⟳'
        else: col,bg,st='#aaa','#f5f5f5','○'
        html+=f'<div style="display:flex;align-items:center;gap:10px;margin:8px 0;background:{bg};border-radius:8px;padding:8px 14px;border-left:3px solid {col};"><span style="font-size:18px;">{icon}</span><div><div style="font-weight:600;color:{col};">{st} {name}</div><div style="color:#666;font-size:12px;">{desc}</div></div></div>'
    if parsed and step>=1:
        html+=f'<div style="margin-top:10px;padding:8px 14px;background:#f0eeff;border-radius:8px;font-size:12px;color:#534AB7;"><b>Detected:</b> {parsed.get("problem_type","?").title()} · {parsed.get("operation","?").title()}</div>'
    if verif and step>=3:
        ok=verif.get('verified',False); bg2='#E1F5EE' if ok else '#FCEBEB'; c2='#1D9E75' if ok else '#A32D2D'
        html+=f'<div style="margin-top:8px;padding:8px 14px;background:{bg2};border-radius:8px;font-size:12px;color:{c2};"><b>{"✓ Verified" if ok else "⚠ Unverified"}</b> — {verif.get("method","")}<br><span style="opacity:0.8;">{verif.get("detail","")}</span></div>'
    html+='</div>'; return html

def make_result_html(result, verif=None):
    if not result.get('success'): return f'<div style="color:#A32D2D;padding:16px;background:#FCEBEB;border-radius:8px;">❌ {result.get("error","Unknown error")}</div>'
    rl=result.get('result_latex',result.get('result_str','See graph/proof'))
    engine=result.get('engine','SymPy'); ok=result.get('verified',True)
    badge_bg='#E1F5EE' if ok else '#FFF3CD'; badge_col='#1D9E75' if ok else '#856404'
    badge_txt='Exact symbolic' if ok else 'Approximate'
    steps_html=''
    for i,s in enumerate(result.get('steps',[]),1):
        steps_html+=f'<div style="display:flex;gap:10px;align-items:flex-start;margin:6px 0;"><div style="min-width:24px;height:24px;background:#534AB7;color:white;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;flex-shrink:0;">{i}</div><div style="font-size:13px;color:#333;padding:5px 10px;background:#f8f7ff;border-radius:6px;width:100%;font-family:monospace;">{s}</div></div>'
    return f'''
    <div style="background:white;border:1px solid #e0dff7;border-radius:12px;overflow:hidden;">
      <div style="background:#534AB7;color:white;padding:10px 16px;font-weight:600;">🎯 Result</div>
      <div style="padding:16px 20px;">
        <div style="font-size:20px;font-family:monospace;color:#222;background:#f8f7ff;padding:14px;border-radius:8px;word-break:break-all;margin-bottom:10px;">{rl}</div>
        <div style="font-size:12px;color:#888;">Engine: <b style="color:#534AB7;">{engine}</b> &nbsp;·&nbsp; <span style="background:{badge_bg};color:{badge_col};padding:2px 10px;border-radius:12px;font-weight:600;">{badge_txt}</span></div>
      </div>
    </div>
    <div style="background:white;border:1px solid #e0dff7;border-radius:12px;overflow:hidden;margin-top:10px;">
      <div style="background:#1D9E75;color:white;padding:10px 16px;font-weight:600;">📐 Engine Steps</div>
      <div style="padding:14px 18px;">{steps_html}</div>
    </div>'''

EXAMPLES = [
    ['Integrate x^2 * sin(x) dx'],
    ['Differentiate x^3 * ln(x)'],
    ['Solve x^3 - 6x^2 + 11x - 6 = 0'],
    ['Find the limit of sin(x)/x as x approaches 0'],
    ['Find the limit of (1+1/n)^n as n approaches infinity'],
    ['Plot sin(x) + cos(2x) from -2pi to 2pi'],
    ['Plot x^3 - 3x from -3 to 3'],
    ['Factor x^4 - 16'],
    ['Expand (x+y)^5'],
    ['Prove that (a+b)^2 = a^2 + 2ab + b^2'],
    ['Prove sin^2(x) + cos^2(x) = 1'],
    ['Find the Taylor series of e^x'],
]

CSS = """
.gradio-container { max-width: 1100px !important; margin: 0 auto !important; }
button.primary { background: #534AB7 !important; color: white !important; border-radius: 10px !important; font-weight: 600 !important; }
"""

def solve_pipeline(query, api_key):
    if not query.strip(): yield make_status(-1),'','',None,'*Enter a problem.*'; return
    if not api_key.strip(): yield make_status(-1),'','',None,'*Enter your Gemini API key.*'; return
    yield make_status(0),'','',None,''
    try:
        parsed=do_parse(query, api_key.strip())
    except Exception as e:
        yield make_status(0),f'<p style="color:red;">Gemini parse error: {e}</p>','',None,''; return
    ahtml=f'<div style="background:white;border:1px solid #e0dff7;border-radius:12px;overflow:hidden;"><div style="background:#534AB7;color:white;padding:10px 16px;font-weight:600;">🔬 Analysis</div><table style="width:100%;font-size:13px;">'+\
          ''.join(f'<tr><td style="padding:6px 12px;color:#888;">{k}</td><td style="padding:6px 12px;font-weight:500;">{v}</td></tr>' for k,v in [('Type',parsed.get('problem_type','').title()),('Operation',parsed.get('operation','').title()),('Variables',', '.join(parsed.get('variables',['x']))),('Domain',parsed.get('domain') or 'real'),('Proof','Yes' if parsed.get('is_proof') else 'No')])+\
          '</table></div>'
    yield make_status(1,parsed=parsed),ahtml,'',None,''
    result=route(parsed)
    graph_img=None
    if 'image_base64' in result:
        import tempfile; img_bytes=base64.b64decode(result['image_base64'])
        tmp=tempfile.NamedTemporaryFile(suffix='.png',delete=False); tmp.write(img_bytes); tmp.close(); graph_img=tmp.name
    rhtml=make_result_html(result)
    yield make_status(2,parsed=parsed),ahtml,rhtml,graph_img,''
    verif=do_verify(parsed,result)
    yield make_status(3,parsed=parsed,verif=verif),ahtml,rhtml,graph_img,''
    explanation=do_explain(query,parsed,result,verif,api_key.strip())
    yield make_status(4,parsed=parsed,verif=verif),ahtml,rhtml,graph_img,explanation

with gr.Blocks(css=CSS, title='NeuroSymbolic Math Solver', theme=gr.themes.Soft()) as demo:
    gr.HTML('''
    <div style="background:linear-gradient(135deg,#534AB7 0%,#3B8BD4 100%);border-radius:16px;padding:28px 36px;margin-bottom:20px;">
      <h1 style="color:white;margin:0 0 8px;font-size:26px;">🧮 NeuroSymbolic Math Solver</h1>
      <p style="color:rgba(255,255,255,0.85);margin:0;font-size:14px;">Gemini (free) understands · SymPy/Z3/Matplotlib computes exactly · Verified before display</p>
    </div>''')
    gr.HTML('<div style="background:#f0eeff;border-left:4px solid #534AB7;border-radius:0 8px 8px 0;padding:12px 18px;margin-bottom:16px;font-size:13px;color:#333;"><b>Pipeline:</b> 🧠 Gemini parses intent → ⚙️ SymPy computes exactly → 🔍 Verifier checks → 📖 Gemini explains · <b>The LLM never does the math</b></div>')
    with gr.Row():
        with gr.Column(scale=2):
            api_key_box=gr.Textbox(label='🔑 Gemini API Key (free at aistudio.google.com)',placeholder='AIza...',type='password',info='1500 free requests/day — no credit card needed')
            query_box=gr.Textbox(label='📝 Math Problem',placeholder='e.g. Integrate x² sin(x) dx   |   Prove (a+b)²=a²+2ab+b²   |   Plot sin(x)/x',lines=3)
            solve_btn=gr.Button('🚀 Solve Problem',variant='primary',size='lg')
            gr.Examples(examples=EXAMPLES,inputs=[query_box],label='Quick examples')
        with gr.Column(scale=1):
            status_out=gr.HTML(make_status(-1))
    with gr.Tabs():
        with gr.Tab('🎯 Answer & Steps'):
            with gr.Row():
                with gr.Column(scale=1): analysis_out=gr.HTML()
                with gr.Column(scale=2): result_out=gr.HTML()
            graph_out=gr.Image(label='📈 Graph',height=400)
        with gr.Tab('📖 Explanation'):
            explanation_out=gr.Markdown(value='*Run the solver to see a detailed explanation.*')
        with gr.Tab('ℹ️ About'):
            gr.Markdown('''
## NeuroSymbolic Math Solver
This project combines neural language understanding with symbolic mathematical computation.
**The LLM (Gemini) never computes math** — it only reads the problem and explains the steps.
All computation is done by exact symbolic engines:
- **SymPy** — algebra, calculus, limits, series, factoring
- **Matplotlib** — function plotting
- **SymPy + Z3** — algebraic proof verification
- **SciPy** — numerical root finding

Results are verified before display (back-substitution, derivative checks, numerical cross-checks).

**Free API used:** Google Gemini 1.5 Flash — 1500 requests/day free at aistudio.google.com
            ''')
    solve_btn.click(fn=solve_pipeline,inputs=[query_box,api_key_box],outputs=[status_out,analysis_out,result_out,graph_out,explanation_out])
    query_box.submit(fn=solve_pipeline,inputs=[query_box,api_key_box],outputs=[status_out,analysis_out,result_out,graph_out,explanation_out])

demo.launch(share=True, server_name='0.0.0.0', server_port=7860, show_api=False)
print('\n🎉 Your public demo link is printed above! Share it for your live demo.')